In [33]:
import supervision as sv
from ultralytics import YOLO
import numpy as np
import matplotlib.pyplot as plt
import urllib.request 
from pathlib import Path
import cv2

Path('assets').mkdir(exist_ok=True)
model = YOLO('yolov8n.pt')
urllib.request.urlretrieve(
    'https://media.roboflow.com/supervision/video-examples/vehicles.mp4',
    'assets/vehiculos.mp4'
)

video = sv.VideoInfo.from_video_path('assets/vehiculos.mp4')

In [34]:
tracker = sv.ByteTrack()

cap = cv2.VideoCapture('assets/vehiculos.mp4')
_, videoFrame = cap.read()
cap.release()

result = model(videoFrame, verbose=False)[0]
pre_tracker = sv.Detections.from_ultralytics(result)
post_tracker = tracker.update_with_detections(pre_tracker)
tracker.reset()

In [35]:
box_annotator = sv.BoxAnnotator()
label_annotator = sv.LabelAnnotator()

def procesar(frame, index):
    result = model(frame, verbose=False)[0]
    pre_tracker = sv.Detections.from_ultralytics(result)
    post_tracker = tracker.update_with_detections(pre_tracker)
    labels = [f'ID: {id}' for id in post_tracker.tracker_id]
    annotated = box_annotator.annotate(scene=frame.copy(), detections=post_tracker)
    annotated = label_annotator.annotate(scene=annotated, detections=post_tracker, labels=labels)
    return annotated

In [36]:
sv.process_video(
    source_path='assets/vehiculos.mp4',
    target_path='assets/vehiculos_identificados.mp4',
    callback=procesar
)

In [37]:
# Procesamos los primeros 3 frames manualmente para ver cómo evolucionan los IDs
tracker.reset()

cap = cv2.VideoCapture("assets/vehicles.mp4")
for frame_num in range(3):
    ret, frame = cap.read()
    if not ret:
        break
    results = model(frame, verbose=False)[0]
    det = sv.Detections.from_ultralytics(results)
    det = tracker.update_with_detections(det)
    print(f"Frame {frame_num}: {len(det)} objetos | IDs: {det.tracker_id}")
cap.release()
# 💭 Reflexión: ¿Los IDs del frame 0 aparecen también en el frame 1 y 2?
# Si un objeto desaparece y vuelve a aparecer después de varios frames,
# el tracker puede asignarle un ID diferente.

In [38]:
tracker.reset()

def callback_clase_id(frame: np.ndarray, _: int) -> np.ndarray:
    results = model(frame, verbose=False)[0]
    det = sv.Detections.from_ultralytics(results)
    det = tracker.update_with_detections(det)
    # Combinar el nombre de la clase con el ID del tracker
    # — más informativo que mostrar solo el ID
    labels = [
        f"{results.names[c]} #{tid}"
        for c, tid in zip(det.class_id, det.tracker_id)
    ]
    scene = box_annotator.annotate(scene=frame.copy(), detections=det)
    return label_annotator.annotate(scene=scene, detections=det, labels=labels)

sv.process_video(
    source_path="assets/vehiculos.mp4",
    target_path="assets/vehicles_clase_id.mp4",
    callback=callback_clase_id
)
print("Guardado: assets/vehicles_clase_id.mp4")
# 💭 Reflexión: ¿Qué label resulta más útil para tu aplicación:
# solo el ID, solo la clase, o ambos?

Guardado: assets/vehicles_clase_id.mp4


In [39]:
# Combinar filtrado (NB03) con tracking — el orden importa:
# filtramos ANTES de pasar al tracker para que no gaste IDs en objetos que no nos interesan
CLASE_OBJETIVO = 2  # 'car' en COCO

tracker.reset()

def callback_solo_autos(frame: np.ndarray, _: int) -> np.ndarray:
    results = model(frame, verbose=False)[0]
    det = sv.Detections.from_ultralytics(results)
    # Filtrar ANTES del tracker — así el tracker solo gestiona la clase de interés
    det = det[det.class_id == CLASE_OBJETIVO]
    det = tracker.update_with_detections(det)
    labels = [f"auto #{tid}" for tid in det.tracker_id]
    scene = box_annotator.annotate(scene=frame.copy(), detections=det)
    return label_annotator.annotate(scene=scene, detections=det, labels=labels)

sv.process_video(
    source_path="assets/vehiculos.mp4",
    target_path="assets/vehicles_autos.mp4",
    callback=callback_solo_autos
)
print("Guardado: assets/vehicles_autos.mp4")
# 💭 Reflexión: ¿Por qué conviene filtrar ANTES de pasar al tracker
# en lugar de filtrar DESPUÉS?

Guardado: assets/vehicles_autos.mp4


In [40]:
# Escribe tu solución aquí
frame_count = {}

tracker.reset()

def mi_callback(frame):
    results = model(frame, verbose=False)[0]
    det = sv.Detections.from_ultralytics(results)
    det = tracker.update_with_detections(det)
    for track_id in det.tracker_id:
        if track_id is not None:
            if track_id not in frame_count:
                frame_count[track_id] = 0
            frame_count[track_id] += 1
    labels = [f'ID: {track_id}, {frame_count[track_id]} objetos' for track_id in det.tracker_id]
    scene = box_annotator.annotate(scene=frame.copy(), detections=det)
    scene = label_annotator.annotate(scene=scene, detections=det, labels=labels)
    print(labels)
    return scene

mi_callback(frame=videoFrame)

['ID: 1, 1 objetos', 'ID: 2, 1 objetos', 'ID: 3, 1 objetos', 'ID: 4, 1 objetos']


array([[[172, 146, 126],
        [172, 146, 126],
        [174, 148, 128],
        ...,
        [180, 155, 132],
        [178, 155, 132],
        [178, 155, 132]],

       [[172, 146, 126],
        [172, 146, 126],
        [174, 148, 128],
        ...,
        [180, 155, 132],
        [178, 155, 132],
        [178, 155, 132]],

       [[172, 146, 126],
        [172, 146, 126],
        [174, 148, 128],
        ...,
        [180, 155, 132],
        [178, 155, 132],
        [178, 155, 132]],

       ...,

       [[104, 105, 110],
        [105, 106, 111],
        [105, 106, 111],
        ...,
        [ 99, 100, 105],
        [ 99, 100, 105],
        [ 99, 100, 105]],

       [[105, 106, 111],
        [105, 106, 111],
        [104, 105, 110],
        ...,
        [ 99, 100, 105],
        [ 99, 100, 105],
        [ 99, 100, 105]],

       [[101, 102, 107],
        [101, 102, 107],
        [101, 102, 107],
        ...,
        [ 99, 100, 105],
        [ 99, 100, 105],
        [ 99, 100, 105]]